<a href="https://colab.research.google.com/github/Loopinlogix/Market_Analysis_Project-2/blob/main/Stock_Market_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stock Market Analysis Project 2


## Intro to this Project

This notebook is all about digging into some historical stock market data. We're gonna do a bunch of things: grab the data, clean it up (get rid of weird errors and bad values), find any crazy outliers, check for duplicates, cook up some new features from the existing data, make sure everything's on the same scale, and then split it all up so we can eventually build some machine learning models.

Basically, we've got two main data files: one with general info about stocks (`historical_stocks.csv`) like where they're traded, their names, what industry they're in, etc., and another with the daily prices and trading volumes (`historical_stock_prices.csv`).

The whole point here is to take all that raw, messy stock info and turn it into something neat and organized, packed with useful features. This way, we'll have a solid dataset ready to go for training models to try and figure out what the stock market might do next.

In [ ]:

#Github

#Github
!apt-get install -y git
!git config --global user.email "crystal_macneil@hotmail.com"
!git config --global user.name "Crystal MacNeil"

!git clone https://github.com/Loopinlogix/Market_Analysis_Project-2.git
%cd Market_Analysis_Project-2
!ls


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'Market_Analysis_Project-2'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/Market_Analysis_Project-2
README.md


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("=" * 75)
print("STEP 1: LOAD AND MERGE THE DATA")
print("=" * 75)

# Load two datasets
stocks = pd.read_csv('historical_stocks.csv')
prices = pd.read_csv('historical_stock_prices.csv')

print(prices.head())
print(stocks.head())
print(prices.info())
print(stocks.info())

# Clean up column names (remove extra spaces, make lowercase)
stocks.columns = stocks.columns.str.strip().str.lower()
prices.columns = prices.columns.str.strip().str.lower()

print(f"Stocks info shape: {stocks.shape}")
print(f"Prices info shape: {prices.shape}")


#remove "fake" header rows.
initial_rows = len(stocks)
stocks = stocks[stocks['ticker'].str.upper() != 'SYMBOL']
stocks = stocks[stocks['ticker'].str.upper() != 'TICKER']
stocks['sector'] = stocks['sector'].replace('N/A', np.nan)
stocks['industry'] = stocks['industry'].replace('N/A', np.nan)
print(f"Removed {initial_rows - len(stocks)} repeated header rows")

# merge the two datasets using the 'ticker' column
df = pd.merge(prices, stocks, on='ticker', how='left')
print(f"Combined dataset shape: {df.shape}")


print("\n" + "=" * 75)
print("STEP 2: FIND AND FIX DATA ERRORS")
print("=" * 75)

# Convert text dates to actual date format
df['date'] = pd.to_datetime(df['date'], errors='coerce')
bad_dates = df['date'].isna().sum()
if bad_dates > 0:
    print(f"Found {bad_dates} bad dates remove them")

# Check for common data problems
errors_found = []

# High should always be >= Low
if (df['high'] < df['low']).any():
    errors_found.append(f"  High < Low: {(df['high'] < df['low']).sum()} rows")

# Close price should be between Low and High
if ((df['close'] < df['low']) | (df['close'] > df['high'])).any():
    errors_found.append(f"  Close outside range: {((df['close'] < df['low']) | (df['close'] > df['high'])).sum()} rows")

# Prices can't be negative
if (df[['open', 'high', 'low', 'close', 'adj_close']] < 0).any().any():
    errors_found.append("  Negative prices found")

# Volume can't be negative
if (df['volume'] < 0).any():
    errors_found.append(f"  Negative volume: {(df['volume'] < 0).sum()} rows")

# Check for duplicate entries
dupes = df.duplicated(subset=['ticker', 'date']).sum()
if dupes > 0:
    errors_found.append(f"  Duplicate entries: {dupes} rows")

if errors_found:
    print("Data problems found:")
    for e in errors_found:
        print(e)
else:
    print("No major data problems found!")

# Make sure High is always the maximum of all prices
df['high'] = df[['high', 'low', 'close', 'open']].max(axis=1)
# Make sure Low is always the minimum of all prices
df['low'] = df[['low', 'high', 'close', 'open']].min(axis=1)

# Replace any negative values with missing (NaN)
for col in ['open', 'high', 'low', 'close', 'adj_close']:
    df.loc[df[col] < 0, col] = np.nan
df.loc[df['volume'] < 0, 'volume'] = np.nan

# Remove rows with bad dates and duplicates
df.dropna(subset=['date'], inplace=True)
df.drop_duplicates(subset=['ticker', 'date'], inplace=True)
print(f"Dataset shape after fixes: {df.shape}")


print("\n" + "=" * 75)
print("STEP 3: FILL IN MISSING VALUES")
print("=" * 75)

# For text columns, fill missing with "Unknown"
for col in ['exchange', 'name', 'sector', 'industry']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Sort by ticker and date can use time-based filling
df.sort_values(by=['ticker', 'date'], inplace=True)

# For numbers, try multiple strategies to fill gaps:
numeric_cols = ['open', 'close', 'adj_close', 'low', 'high', 'volume']

for col in numeric_cols:
    # Strategy 1: Use the previous day's value (forward fill)
    df[col] = df.groupby('ticker')[col].transform(lambda x: x.ffill())
    # Strategy 2: Use the next day's value (backward fill)
    df[col] = df.groupby('ticker')[col].transform(lambda x: x.bfill())
    # Strategy 3: Draw a line between known points (linear interpolation)
    df[col] = df.groupby('ticker')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    # Strategy 4: Use the median for that specific stock
    df[col] = df.groupby('ticker')[col].transform(lambda x: x.fillna(x.median()))
    # Strategy 5: Last resort — use the overall median
    df[col].fillna(df[col].median(), inplace=True)

print("Missing values after filling:")
print(df[numeric_cols].isnull().sum())


print("\n" + "=" * 75)
print("STEP 4: HANDLE EXTREME VALUES (OUTLIERS)")
print("=" * 75)

# For each stock individually, cap extreme values using the IQR method
# IQR = range between the 25th and 75th percentiles
for col in ['open', 'high', 'low', 'close', 'volume']:
    q1 = df.groupby('ticker')[col].transform(lambda x: x.quantile(0.25))
    q3 = df.groupby('ticker')[col].transform(lambda x: x.quantile(0.75))
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outliers = ((df[col] < lower_limit) | (df[col] > upper_limit)).sum()
    df[col] = df[col].clip(lower=lower_limit, upper=upper_limit)
    print(f"  {col}: Capped {outliers} extreme values per stock")


print("\n" + "=" * 75)
print("STEP 5: CREATE NEW FEATURES")
print("=" * 75)

df.sort_values(by=['ticker', 'date'], inplace=True)

# Moving averages (smooth out daily noise)
df['rolling_close_7'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(7).mean())
df['rolling_close_30'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(30).mean())

# Volatility (how much the price swings around)
df['volatility_7'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(7).std())
df['volatility_30'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(30).std())

# Daily return (percentage change from yesterday)
df['daily_return'] = df.groupby('ticker')['close'].pct_change()

# Price range (how much the stock moved that day)
df['price_range'] = df['high'] - df['low']

# Simple and Exponential Moving Averages
df['sma_14'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(14).mean())
df['ema_14'] = df.groupby('ticker')['close'].transform(lambda x: x.ewm(span=14, adjust=False).mean())

print("Features created successfully!")


print("\n" + "=" * 75)
print("STEP 6: PREPARE FINAL DATASET")
print("=" * 75)

# Drop rows that still have missing values (usually from rolling calculations)
df_model = df.dropna().copy()
print(f"Final dataset shape: {df_model.shape}")
print(f"Rows dropped: {len(df) - len(df_model)}")


print("\n" + "=" * 75)
print("STEP 7: CONVERT TEXT CATEGORIES TO NUMBERS")
print("=" * 75)

# One-hot encoding: turn categories like "Tech", "Finance" into 0/1 columns
df_model_encoded = pd.get_dummies(
    df_model,
    columns=['exchange', 'sector', 'industry'],
    drop_first=True
)
print(f"Shape after encoding: {df_model_encoded.shape}")


print("\n" + "=" * 75)
print("STEP 8: SPLIT INTO TRAIN / VALIDATION / TEST")
print("=" * 75)

# IMPORTANT: split BEFORE scaling to prevent data leakage
target = 'close'
X = df_model_encoded.drop(columns=[target, 'ticker', 'date', 'name'])
y = df_model_encoded[target]

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, shuffle=True
)
# Second split: split the 30% into 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, shuffle=True
)

print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")


print("\n" + "=" * 75)
print("STEP 9: SCALE THE NUMBERS")
print("=" * 75)

# StandardScaler makes all numbers have mean=0 and std=1
# ONLY fit on training data, then apply to validation and test
numeric_cols_for_scaling = X_train.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
X_train[numeric_cols_for_scaling] = scaler.fit_transform(X_train[numeric_cols_for_scaling])
X_val[numeric_cols_for_scaling] = scaler.transform(X_val[numeric_cols_for_scaling])
X_test[numeric_cols_for_scaling] = scaler.transform(X_test[numeric_cols_for_scaling])

print(f"Scaler fit on training data only (no data leakage!)")
print(f"  Training mean: {X_train[numeric_cols_for_scaling].mean().mean():.6f}")
print(f"  Training std:  {X_train[numeric_cols_for_scaling].std().mean():.6f}")


print("\n" + "=" * 75)
print("STEP 10: SAVE EVERYTHING")
print("=" * 75)

df_model.to_csv('/content/clean_stock_data.csv', index=False)
X_train.to_csv('/content/X_train.csv', index=False)
X_val.to_csv('/content/X_val.csv', index=False)
X_test.to_csv('/content/X_test.csv', index=False)
y_train.to_csv('/content/y_train.csv', index=False)
y_val.to_csv('/content/y_val.csv', index=False)
y_test.to_csv('/content/y_test.csv', index=False)

print("\n" + "=" * 75)
print("ALL DONE! Here's what we accomplished:")
print("=" * 75)
print("""
Data Loaded & Merged: Started with millions of raw stock records.
Data Cleaned: Fixed bad dates, extreme values, and removed duplicates.
Missing Values Filled: No more gaps in our data!
Outliers Handled: Extreme price/volume movements were gently capped.
Features Engineered: Created powerful new indicators like moving averages and volatility.
Dataset Prepared: Dropped remaining NaNs from rolling calculations.
Categorical Data Encoded: Text labels converted to numbers for machine learning.
Data Split: Divided into training, validation, and test sets for robust model evaluation.
Numeric Data Scaled: Normalized values for optimal model performance.
All Data Saved: Cleaned data and splits are ready for the next steps!""")


STEP 1: LOAD AND MERGE THE DATA
  ticker   open  close  adj_close    low   high   volume        date
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900  2013-05-08
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800  2013-05-09
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100  2013-05-10
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400  2013-05-13
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100  2013-05-14
  ticker exchange                                    name             sector  \
0    PIH   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
1  PIHPP   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
2   TURN   NASDAQ                180 DEGREE CAPITAL CORP.            FINANCE   
3   FLWS   NASDAQ                 1-800 FLOWERS.COM, INC.  CONSUMER SERVICES   
4   FCCY   NASDAQ           1ST CONSTITUTION BANCORP (NJ)            FINANCE   

                     industry  
0  PROPERTY-CASUALTY INSURERS  
1  PROPER